[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VincentALBoyer/Course-Materials/blob/main/Undergraduate%20Level/Metaheuristics/JupyterNotebooks/05_Escaping_Local_Optima.ipynb)

# IN2041 · Session 5 — Escaping Local Optima (interactive lab)

Local search stops at the first solution that none of its neighbors improve: a **local optimum**.
This notebook lets you *see* those traps and play with the four escape strategies of the session:

| # | Strategy | What it does when we are stuck |
|---|----------|--------------------------------|
| 1 | **Multi-start** | throw the solution away and restart from a new random solution |
| 2 | **Perturbation (ILS)** | apply *k* random moves to the current local optimum |
| 3 | **Destroy & Repair** | remove *k* elements and rebuild them greedily |
| 4 | **Accept worse** | make one deliberately worse move, and sometimes keep a worse local optimum |

**Contents**
0. Setup
1. Warm-up: the Startup Investment Auction (opening activity)
2. Knapsack: what a local optimum looks like
3. One escape loop for every problem
4. Knapsack: escape strategies in action
5. p-Median: traps on a map
6. p-Median: escape strategies in action
7. Takeaways and exercises

> **Hidden code:** every code cell is collapsed: you only see its title, and it still runs with *Run all*. Click a title (Colab: *Show code*) to read it. The cells marked **(worth reading)** contain the knapsack and p-median **moves** and the **escape loop** discussed in class.
>
> **How to use it:** run the cells from top to bottom (*Run All*). Cells with sliders and drop-down menus are interactive:
> change a value and the figure updates. If `ipywidgets` is not installed, every figure is still drawn with its default values.

## 0. Setup

In [ ]:
# @title Setup: libraries and widget helpers (run me first) { display-mode: "form" }
%matplotlib inline
import itertools, math, functools
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:
    import ipywidgets as W
    HAS_WIDGETS = True
except ImportError:
    # Minimal stand-in: every interactive cell is drawn once with its default values.
    HAS_WIDGETS = False
    print("ipywidgets not found -> static figures. To get the sliders run:  %pip install ipywidgets")
    class _Control:
        def __init__(self, *args, value=None, options=None, **kwargs):
            self.value = value if value is not None else (options[0] if options else None)
    class W:
        Dropdown = IntSlider = FloatSlider = Checkbox = _Control
        @staticmethod
        def interact(f, **controls):
            f(**{name: getattr(c, "value", c) for name, c in controls.items()})
        interact_manual = interact

def slider(lo, hi, value, step=1, desc=""):
    """Integer slider (or float slider if step is a float)."""
    cls = W.FloatSlider if isinstance(step, float) else W.IntSlider
    return cls(min=lo, max=hi, step=step, value=value, description=desc,
               continuous_update=False, style={"description_width": "initial"})

def dropdown(options, value, desc=""):
    return W.Dropdown(options=options, value=value, description=desc, style={"description_width": "initial"})

plt.rcParams.update({"figure.figsize": (11, 4.2), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
COLORS = {"current": "#1f77b4", "best": "#d62728", "opt": "#2ca02c", "kick": "#9e9e9e",
          "greedy": "#ff7f0e", "trap": "#d62728"}
INSTANCES, OPT = {}, {}      # registry used by the interactive explorers

## 1. Warm-up: the Startup Investment Auction

Same data as the opening activity of the slides: pick **5 of 30 startups**, maximize base scores **plus** synergy bonuses.
A bonus is earned only when **all three** startups of a trio are in the portfolio.

In [ ]:
# @title Startup auction: data and helper functions { display-mode: "form" }
STARTUPS = ["QuantumLeap AI", "SolarNova Energy", "BioPulse Health", "CloudForge Systems", "NeuroNest Robotics",
            "GreenGrid Infra", "FinFlare Analytics", "DeepTide Security", "AeroSpark Mobility", "EcoSphere Farming",
            "CodeCrush DevTools", "MedMosaic Diagnostics", "SkyChain Logistics", "NanoNest Materials", "TerraLink Satellites",
            "MindMesh Education", "AquaBloom WaterTech", "SynthWave MusicTech", "VibeVault Wearables", "RoboRoot Automation",
            "ZenByte Wellness", "CryptoNest Finance", "DriftDrive EVs", "Plantify Biotech", "DataDrill Mining",
            "LoopLogic Recycling", "PulsePoint IoT", "BrightBridge EdTech", "MetaMorph AR", "SparkSphere Events"]
SCORES = [95, 90, 88, 85, 83, 82, 80, 78, 76, 75, 74, 72, 70, 68, 66,
          65, 64, 62, 60, 58, 56, 54, 52, 50, 48, 46, 44, 42, 40, 38]
SYNERGIES = [((7, 8, 11), 70, "Analytics + Security + DevTools"),
             ((2, 6, 10), 50, "Green Energy + Infra + Farming"),
             ((3, 12, 24), 45, "Health + Diagnostics + Biotech"),
             ((5, 9, 20), 40, "Robotics + Mobility + Automation"),
             ((16, 18, 28), 35, "Education + Music + EdTech"),
             ((13, 15, 23), 30, "Logistics + Satellites + EVs"),
             ((8, 14, 27), 25, "Security + Materials + IoT")]
BUDGET = 5

def portfolio_value(port):
    port = set(port)
    return sum(SCORES[i - 1] for i in port) + sum(b for trio, b, _ in SYNERGIES if set(trio) <= port)

def best_k_swap(port, k):
    """Best portfolio obtained by replacing exactly k startups. Returns (None, value) if none improves."""
    port, best, best_val = set(port), None, portfolio_value(port)
    others = [i for i in range(1, 31) if i not in port]
    for out in itertools.combinations(sorted(port), k):
        for inn in itertools.combinations(others, k):
            cand = (port - set(out)) | set(inn)
            v = portfolio_value(cand)
            if v > best_val:
                best, best_val = (out, inn, cand), v
    return best, best_val

def fmt(port):
    return ", ".join(f"{i:02d}" for i in sorted(port))

### 1.1 Build your own portfolio
Toggle startups on and off. The panel tells you whether your portfolio is a **local optimum** for 1-, 2- and 3-swap moves
(tick the box to reveal the improving swap, spoiler!).

In [ ]:
# @title Interactive portfolio builder { display-mode: "form" }
def report(port, show_hints):
    port = set(port)
    base = sum(SCORES[i - 1] for i in port)
    bonus = [(trio, b, name) for trio, b, name in SYNERGIES if set(trio) <= port]
    print(f"Portfolio: {fmt(port)}")
    print(f"Base score: {base}" + "".join(f"\n  + {b} synergy {name} {list(trio)}" for trio, b, name in bonus))
    print(f"TOTAL VALUE: {portfolio_value(port)}\n")
    if len(port) != BUDGET:
        print(f"Select exactly {BUDGET} startups (now {len(port)}).")
        return
    for k in (1, 2, 3):
        move, val = best_k_swap(port, k)
        if move is None:
            print(f"  {k}-swap: no improving move  -> LOCAL OPTIMUM for {k}-swaps")
        elif show_hints:
            out, inn, _ = move
            print(f"  {k}-swap: replace {list(out)} by {list(inn)}  -> {val} (+{val - portfolio_value(port)})")
        else:
            print(f"  {k}-swap: an improving move exists")

if HAS_WIDGETS:
    buttons = [W.ToggleButton(description=f"{i + 1:02d} {STARTUPS[i]} ({SCORES[i]})", value=i < BUDGET,
                              layout=W.Layout(width="235px")) for i in range(30)]
    hints = W.Checkbox(description="Show the improving swaps (spoiler!)", value=False)
    out = W.Output()
    def refresh(_=None):
        with out:
            out.clear_output()
            report({i + 1 for i, b in enumerate(buttons) if b.value}, hints.value)
    for b in buttons + [hints]:
        b.observe(refresh, "value")
    display(W.VBox([W.GridBox(buttons, layout=W.Layout(grid_template_columns="repeat(4, 240px)")), hints, out]))
    refresh()
else:
    report(range(1, 6), show_hints=True)

### 1.2 The debrief: a ladder of local optima
Start from the greedy portfolio (top 5 scores) and climb with best-improvement local search, first with 1-swap moves,
then 2-swap, then 3-swap. **Each rung is a local optimum for the moves allowed so far.**

In [ ]:
# @title Climb the ladder of local optima { display-mode: "form" }
port = set(range(1, 6))
rows = [("Greedy (top 5 scores)", fmt(port), portfolio_value(port))]
for k in (1, 2, 3):
    while True:                                   # best-improvement local search with k-swap moves
        move, _ = best_k_swap(port, k)
        if move is None:
            break
        port = move[2]
    rows.append((f"+ local search with {k}-swap moves", fmt(port), portfolio_value(port)))

display(pd.DataFrame(rows, columns=["Strategy", "Portfolio", "Value"]))
fig, ax = plt.subplots(figsize=(9, 3))
ax.barh([r[0] for r in rows][::-1], [r[2] for r in rows][::-1], color=["#2ca02c", "#1f77b4", "#1f77b4", "#ff7f0e"])
for y, r in enumerate(rows[::-1]):
    ax.text(r[2] + 1, y, str(r[2]), va="center")
ax.set_xlim(400, 500); ax.set_xlabel("portfolio value"); ax.set_title("Bigger moves escape deeper traps")
plt.show()

## 2. Knapsack: what a local optimum looks like

### 2.1 The knapsack toolbox
A solution is a 0/1 vector `x` (1 = item packed). The class below gathers everything the heuristics need:

* **construction**: `greedy` (by benefit/weight ratio, by benefit or lightest first) and `random_start` (a *randomized* greedy: ratios multiplied by random noise, the "diversified initial solutions" of the slides)
* **local search** (best improvement): the *1-flip* neighborhood (add or remove one item), optionally enlarged with *swap* moves (one item out, one in)
* **escape moves**: `perturb` (flip *k* random bits, then repair), `destroy_repair` (remove *k* items, refill greedily), `worse_neighbor` (a random exchange that loses benefit)
* `optimal`: exact dynamic programming, **only** to measure how far the heuristics are from the optimum

In [ ]:
# @title Knapsack toolbox: greedy, local search and escape moves (worth reading) { display-mode: "form" }
class KnapsackProblem:
    """0-1 knapsack: maximize p·x subject to w·x <= C.  A solution is a 0/1 numpy vector."""
    sense = "max"

    def __init__(self, name, items, profits, weights, capacity, scale=1):
        self.name, self.items = name, list(items)
        self.p = np.asarray(profits, dtype=float)
        self.w = np.asarray(weights, dtype=float)
        self.C = float(capacity)
        self.n = len(self.p)
        self.ratio = self.p / self.w
        self.scale = scale          # weights * scale must be integers (used by the exact DP)
        self.neighborhood = "1-flip"  # or "1-flip + swap"

    # ---------- evaluation ----------
    def value(self, x):  return float(self.p @ x)
    def weight(self, x): return float(self.w @ x)
    def feasible(self, x): return self.weight(x) <= self.C + 1e-9
    def describe(self, x):
        return "{" + ", ".join(self.items[i] for i in np.flatnonzero(x)) + "}"

    # ---------- construction ----------
    def greedy(self, criterion="ratio", banned=(), x=None):
        """Add items in decreasing order of the criterion while they fit."""
        score = {"ratio": self.ratio, "profit": self.p, "light": -self.w}[criterion]
        x = np.zeros(self.n, dtype=int) if x is None else x.copy()
        free = self.C - self.weight(x)
        for i in np.argsort(-score, kind="stable"):
            if x[i] == 0 and i not in banned and self.w[i] <= free + 1e-9:
                x[i], free = 1, free - self.w[i]
        return x

    def random_start(self, rng):
        """Diversified start (randomized greedy): ratio-greedy on noisy scores, ratio x U(0.5, 1.5)."""
        noisy = self.ratio * rng.uniform(0.5, 1.5, self.n)
        x, free = np.zeros(self.n, dtype=int), self.C
        for i in np.argsort(-noisy):
            if self.w[i] <= free + 1e-9:
                x[i], free = 1, free - self.w[i]
        return x

    def repair(self, x):
        """Drop the least efficient items until the solution is feasible."""
        x = x.copy()
        for i in np.argsort(self.ratio):
            if self.feasible(x): break
            x[i] = 0
        return x

    # ---------- local search (best improvement) ----------
    def best_move(self, x):
        """Best improving neighbor of x, or None if x is a local optimum."""
        free = self.C - self.weight(x)
        best_gain, best = 1e-9, None
        # 1-flip: only adding an item can improve (removing always loses profit)
        cand = np.flatnonzero((x == 0) & (self.w <= free + 1e-9))
        if cand.size:
            i = cand[np.argmax(self.p[cand])]
            best_gain, best = self.p[i], ("add", i, None)
        if self.neighborhood == "1-flip + swap":
            ins, outs = np.flatnonzero(x == 1), np.flatnonzero(x == 0)
            if ins.size and outs.size:
                gain = self.p[outs][None, :] - self.p[ins][:, None]
                fits = (self.w[outs][None, :] - self.w[ins][:, None]) <= free + 1e-9
                gain = np.where(fits, gain, -np.inf)
                a, b = np.unravel_index(np.argmax(gain), gain.shape)
                if gain[a, b] > best_gain:
                    best_gain, best = gain[a, b], ("swap", ins[a], outs[b])
        return best

    def apply(self, x, move):
        x = x.copy()
        kind, i, j = move
        if kind == "add": x[i] = 1
        else: x[i], x[j] = 0, 1
        return x

    def local_search(self, x, path=None):
        x = x.copy()
        while (move := self.best_move(x)) is not None:
            x = self.apply(x, move)
            if path is not None: path.append(x.copy())
        return x

    # ---------- escape moves ----------
    def perturb(self, x, k, rng):
        """Flip k random bits, then repair if the knapsack overflows."""
        x = x.copy()
        idx = rng.choice(self.n, size=min(k, self.n), replace=False)
        x[idx] = 1 - x[idx]
        return self.repair(x)

    def destroy_repair(self, x, k, rng):
        """Remove k random packed items, refill greedily (removed items are banned during the refill)."""
        packed = np.flatnonzero(x)
        if packed.size == 0: return self.greedy()
        out = rng.choice(packed, size=min(k, packed.size), replace=False)
        y = x.copy(); y[out] = 0
        return self.greedy("ratio", banned=set(out.tolist()), x=y)

    def worse_neighbor(self, x, rng):
        """A deliberately worse move: a random exchange (one item out, a less profitable one in).
        If no such exchange fits, simply take a random item out."""
        ins, outs = np.flatnonzero(x == 1), np.flatnonzero(x == 0)
        free = self.C - self.weight(x)
        pairs = [(i, j) for i in ins for j in outs
                 if self.p[j] < self.p[i] and self.w[j] - self.w[i] <= free + 1e-9]
        y = x.copy()
        if pairs:
            i, j = pairs[rng.integers(len(pairs))]
            y[i], y[j] = 0, 1
        elif ins.size:
            y[rng.choice(ins)] = 0
        return y

    # ---------- exact solution (dynamic programming, for reference only) ----------
    def optimal(self):
        W = np.rint(self.w * self.scale).astype(int); C = int(math.floor(self.C * self.scale + 1e-9))
        best = np.zeros(C + 1); keep = np.zeros((self.n, C + 1), dtype=bool)
        for i in range(self.n):
            cand = np.full(C + 1, -np.inf)
            cand[W[i]:] = best[:C + 1 - W[i]] + self.p[i]
            keep[i] = cand > best
            best = np.maximum(best, cand)
        x, c = np.zeros(self.n, dtype=int), int(np.argmax(best))
        for i in range(self.n - 1, -1, -1):
            if keep[i, c]: x[i], c = 1, c - W[i]
        return x

### 2.2 The whole landscape of a tiny instance
The 5-item example of the slides: benefits $p=[10,40,30,50,25]$, weights $w=[5,4,6,3,2]$, capacity $C=10$.
With only $2^5 = 32$ solutions we can draw **all** of them: x-axis = weight, y-axis = benefit.
Grey lines are 1-flip moves; **red circles** are local optima (no 1-flip improves them); the **gold star** is the global optimum.

In [ ]:
# @title Toy instance: enumerate every solution { display-mode: "form" }
TOY = KnapsackProblem("Slides example", list("ABCDE"), [10, 40, 30, 50, 25], [5, 4, 6, 3, 2], 10)
ALL_TOY = [np.array(b) for b in itertools.product([0, 1], repeat=TOY.n)]
FEAS_TOY = [x for x in ALL_TOY if TOY.feasible(x)]
LO_TOY = [x for x in FEAS_TOY if TOY.best_move(x) is None]
TOY_OPT = TOY.value(TOY.optimal())

print(f"{len(FEAS_TOY)} feasible solutions, {len(LO_TOY)} of them are local optima for 1-flip:")
display(pd.DataFrame([(TOY.describe(x), TOY.weight(x), TOY.value(x), "global" if TOY.value(x) == TOY_OPT else "")
                      for x in sorted(LO_TOY, key=TOY.value, reverse=True)],
                     columns=["Local optimum", "Weight", "Benefit", ""]))

**Try it.** Choose a starting solution: local search climbs (blue arrows) until it is stuck.
Then *accept a worse move* by taking one item out (orange) and let local search climb again (green).
The slides start from {A, B} = 50: compare removing **A** with removing **B**.

In [ ]:
# @title Interactive landscape of the toy instance { display-mode: "form" }
def arrow(ax, x, y, color, ls="-"):
    ax.annotate("", xy=(TOY.weight(y), TOY.value(y)), xytext=(TOY.weight(x), TOY.value(x)),
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.2, ls=ls, shrinkA=6, shrinkB=6))

LABEL_POS = {"{B, D, E}": (-14, 4, "right"), "{C, D}": (-12, -14, "right"),   # label offsets that avoid overlaps
             "{C, E}": (-12, 4, "right"), "{A, B}": (0, -20, "center")}

def toy_landscape(start, remove):
    x0 = np.array(start)
    fig, ax = plt.subplots(figsize=(10, 5.2))
    for x in FEAS_TOY:                                   # 1-flip edges between feasible solutions
        for i in range(TOY.n):
            y = x.copy(); y[i] = 1
            if x[i] == 0 and TOY.feasible(y):
                ax.plot([TOY.weight(x), TOY.weight(y)], [TOY.value(x), TOY.value(y)], color="#e0e0e0", lw=1, zorder=0)
    inf = [x for x in ALL_TOY if not TOY.feasible(x)]
    ax.scatter([TOY.weight(x) for x in inf], [TOY.value(x) for x in inf], marker="x", color="#c8c8c8", label="infeasible")
    ax.scatter([TOY.weight(x) for x in FEAS_TOY], [TOY.value(x) for x in FEAS_TOY], color="#90a4ae", s=30, label="feasible")
    for x in LO_TOY:
        glob = TOY.value(x) == TOY_OPT
        ax.scatter(TOY.weight(x), TOY.value(x), s=420 if glob else 180, marker="*" if glob else "o",
                   facecolor="gold" if glob else "none", edgecolor=COLORS["trap"], lw=2, zorder=3)
        dx, dy, ha = LABEL_POS.get(TOY.describe(x), (10, 4, "left"))
        ax.annotate(f"{TOY.describe(x)} = {TOY.value(x):g}", (TOY.weight(x), TOY.value(x)), textcoords="offset points",
                    xytext=(dx, dy), ha=ha, fontsize=9, color=COLORS["trap"])
    ax.axvline(TOY.C, ls=":", color="k"); ax.text(TOY.C + 0.2, 5, "capacity", rotation=90)

    path = [x0]; x1 = TOY.local_search(x0, path)
    for a, b in zip(path, path[1:]):
        arrow(ax, a, b, COLORS["current"])
    msg = f"start {TOY.describe(x0)} = {TOY.value(x0):g}  ->  local search  ->  {TOY.describe(x1)} = {TOY.value(x1):g}"
    if remove != "(none)":
        i = TOY.items.index(remove)
        if x1[i] == 0:
            msg += f"\n{remove} is not in the knapsack: choose another item to remove."
        else:
            x2 = x1.copy(); x2[i] = 0
            arrow(ax, x1, x2, COLORS["greedy"], ls="--")
            path2 = [x2]; x3 = TOY.local_search(x2, path2)
            for a, b in zip(path2, path2[1:]):
                arrow(ax, a, b, COLORS["opt"])
            msg += (f"\naccept worse: remove {remove} -> {TOY.value(x2):g}  ->  local search  ->  "
                    f"{TOY.describe(x3)} = {TOY.value(x3):g}" + ("   GLOBAL OPTIMUM!" if TOY.value(x3) == TOY_OPT else ""))
    ax.set_xlabel("total weight"); ax.set_ylabel("total benefit"); ax.set_xlim(-0.5, 21); ax.legend(loc="upper left")
    ax.set_title(msg, fontsize=10, loc="left")
    plt.show()

start_options = [(f"{TOY.describe(x)}  (w={TOY.weight(x):g}, p={TOY.value(x):g})", tuple(int(v) for v in x))
                 for x in sorted(FEAS_TOY, key=lambda x: (x.sum(), TOY.describe(x)))]
W.interact(toy_landscape,
           start=dropdown(start_options, (1, 1, 0, 0, 0), "start solution"),
           remove=dropdown(["(none)"] + TOY.items, "A", "then remove item"))

### 2.3 A real instance: the hiker's backpack (slides)
15 items, capacity 10 kg. Which greedy criterion is best? Can local search fix the greedy solution?
Does a **larger neighborhood** (1-flip + swap) help?

In [ ]:
# @title Instances and the interactive hiker view { display-mode: "form" }
HIKER = KnapsackProblem(
    "Hiker (15 items)",
    ["Water", "Chocolate", "Gatorade", "Camera", "Raincoat", "Towel", "First aid kit", "Lantern",
     "Fruit", "Sandwiches", "Tacos", "Sleeping bag", "Book", "Sunscreen", "Champagne"],
    [7, 10, 7, 6, 3, 5, 10, 7, 8, 5, 8, 4, 8, 9, 5],
    [2.00, 0.25, 1.10, 2.00, 0.75, 0.80, 3.00, 1.50, 2.50, 0.50, 0.75, 2.30, 1.30, 0.45, 1.10],
    capacity=10, scale=100)

# A harder random instance (40 items, weights strongly linked to benefits)
_rng = np.random.default_rng(11)
_w = _rng.integers(20, 121, 40)
_p = np.maximum(_w + _rng.integers(-25, 26, 40), 5)
RANDOM40 = KnapsackProblem("Random (40 items)", [f"I{i + 1}" for i in range(40)], _p, _w, int(_rng.integers(300, 500)))

for prob in (HIKER, RANDOM40):
    INSTANCES[prob.name], OPT[prob.name] = prob, prob.value(prob.optimal())
    print(f"{prob.name:<20} optimum = {OPT[prob.name]:g}")

def hiker_view(criterion, neighborhood):
    HIKER.neighborhood = neighborhood
    g = HIKER.greedy(criterion)
    path = [g]; ls = HIKER.local_search(g, path)
    opt = HIKER.optimal()
    table = pd.DataFrame({"Weight": HIKER.w, "Benefit": HIKER.p, "B/W": HIKER.ratio.round(2),
                          "Greedy": np.where(g, "x", ""), "Greedy+LS": np.where(ls, "x", ""),
                          "Optimal": np.where(opt, "x", "")}, index=HIKER.items)
    fig, ax = plt.subplots(figsize=(7, 2.6))
    vals = [HIKER.value(g), HIKER.value(ls), HIKER.value(opt)]
    ax.barh(["Greedy", "Greedy + LS", "Optimal"][::-1], vals[::-1], color=[COLORS["opt"], COLORS["current"], COLORS["greedy"]])
    for y, v in enumerate(vals[::-1]):
        ax.text(v + 0.3, y, f"{v:g}", va="center")
    ax.set_xlim(50, 75); ax.set_title(f"greedy by {criterion}, {neighborhood} local search ({len(path) - 1} moves)")
    plt.show()
    display(table)

W.interact(hiker_view,
           criterion=dropdown(["ratio", "profit", "light"], "ratio", "greedy criterion"),
           neighborhood=dropdown(["1-flip", "1-flip + swap"], "1-flip", "LS neighborhood"))

> **Why does 1-flip local search not improve the greedy solution?** With 1-flip, the only improving move is *adding*
> an item that still fits. A greedy solution is already "full", so it is *immediately* a local optimum.
> To do better we must either enlarge the neighborhood (swap) or **escape**.

## 3. One escape loop for every problem

All four strategies of the session share the same skeleton (compare with the ILS and LNS pseudocode in the slides):

```
x  <- LocalSearch(Greedy())                # a first local optimum
best <- x
repeat:
    y  <- Escape(x)                        # 1. leave the local optimum   (the four strategies differ only here...)
    y  <- LocalSearch(y)                   # 2. climb again
    if Accept(y, x):  x <- y               # 3. move?                     (...and here)
    if y better than best:  best <- y
```

| Strategy | `Escape(x)` | `Accept(y, x)` | Parameter |
|---|---|---|---|
| Multi-start | ignore `x`, new diversified start | always | — |
| Perturbation (ILS) | `k` random moves | if `y` is not worse | `k` = strength |
| Destroy & Repair | remove `k` elements, greedy rebuild | if `y` is not worse | `k` = % destroyed |
| Accept worse | one worse move | if not worse, **or** with probability `q` | `q` |

The function below is **problem-independent**: it only calls the methods `greedy`, `local_search`, `random_start`,
`perturb`, `destroy_repair`, `worse_neighbor` and `value`. Knapsack and p-median just provide their own versions.

In [ ]:
# @title The escape loop: one function for every problem (worth reading) { display-mode: "form" }
STRATEGIES = {
    "Multi-start":         "new random start → LS; keep the best",
    "Perturbation (ILS)":  "k random moves → LS; accept if not worse",
    "Destroy & Repair":    "remove k elements, rebuild greedily → LS; accept if not worse",
    "Accept worse":        "one worse move → LS; accept a worse local optimum with probability q",
}

def run_escape(prob, strategy, iters=100, k=2, q=0.3, seed=0):
    """Generic escape loop. Only the *moves* are problem-specific; the loop is the same for every problem."""
    rng = np.random.default_rng(seed)
    better = (lambda a, b: a > b + 1e-9) if prob.sense == "max" else (lambda a, b: a < b - 1e-9)

    x = prob.local_search(prob.greedy())          # start: greedy + local search = a local optimum
    fx = prob.value(x)
    best, fbest = x.copy(), fx
    T = dict(current=[fx], best=[fbest], sols=[x.copy()], kicked=[fx], accepted=[True])

    for _ in range(iters):
        # 1) ESCAPE: leave the current local optimum
        if strategy == "Multi-start":          y = prob.random_start(rng)
        elif strategy == "Perturbation (ILS)": y = prob.perturb(x, k, rng)
        elif strategy == "Destroy & Repair":   y = prob.destroy_repair(x, k, rng)
        elif strategy == "Accept worse":       y = prob.worse_neighbor(x, rng)
        else: raise ValueError(strategy)
        fkick = prob.value(y)
        # 2) INTENSIFY: local search from the new point
        y = prob.local_search(y)
        fy = prob.value(y)
        # 3) ACCEPT?
        if strategy == "Multi-start":    accept = True
        elif strategy == "Accept worse": accept = (not better(fx, fy)) or rng.random() < q
        else:                            accept = not better(fx, fy)
        if accept: x, fx = y, fy
        if better(fy, fbest): best, fbest = y.copy(), fy
        T["current"].append(fx); T["best"].append(fbest); T["sols"].append(x.copy())
        T["kicked"].append(fkick); T["accepted"].append(accept)
    T["best_sol"] = best
    return T


In [ ]:
# @title Helpers: caching, plots, summary and comparison { display-mode: "form" }
@functools.lru_cache(maxsize=512)
def cached_run(instance, neighborhood, strategy, iters, k, q, seed):
    prob = INSTANCES[instance]
    if hasattr(prob, "neighborhood"):
        prob.neighborhood = neighborhood
    return run_escape(prob, strategy, iters, k, q, seed)

def gap(value, opt):
    return 100 * abs(value - opt) / abs(opt)

def plot_trajectory(T, opt, ax, upto=None, title=""):
    it = np.arange(len(T["current"]))
    core = np.r_[T["current"], T["best"], opt]
    pad = 0.25 * (core.max() - core.min() + 1e-9)
    ax.plot(it, T["kicked"], ".", color=COLORS["kick"], ms=5, label="after the escape move (before LS)")
    ax.step(it, T["best"], where="post", color=COLORS["best"], lw=4, alpha=0.45, label="best found")
    ax.step(it, T["current"], where="post", color=COLORS["current"], lw=1.4, label="current local optimum")
    ax.axhline(opt, ls="--", color=COLORS["opt"], label=f"optimum = {opt:g}")
    if upto is not None:
        ax.axvline(upto, color="k", lw=1, alpha=0.5)
    ax.set_ylim(core.min() - pad, core.max() + pad)
    ax.set_xlabel("iteration  (one escape + one local search)"); ax.set_title(title, loc="left", fontsize=10)
    ax.legend(fontsize=8, loc="best")

def summary(T, opt):
    final = T["best"][-1]
    first_hit = int(np.argmax(np.isclose(T["best"], final)))
    visited = len({tuple(s) for s in T["sols"]})
    print(f"best found = {final:g}   gap to optimum = {gap(final, opt):.2f}%   (found at iteration {first_hit})")
    print(f"start (greedy + LS) = {T['best'][0]:g}   distinct local optima visited = {visited}   "
          f"accepted moves = {sum(T['accepted'][1:])}/{len(T['accepted']) - 1}")

def compare_strategies(instance, neighborhood, iters, seeds, k, q):
    """Run every strategy from `seeds` different random seeds with the same budget of iterations."""
    opt = OPT[instance]
    finals, curves, rows = [], [], []
    for s in STRATEGIES:
        B = np.array([cached_run(instance, neighborhood, s, iters, k, q, sd)["best"] for sd in range(seeds)])
        G = gap(B, opt)
        finals.append(G[:, -1]); curves.append(G.mean(axis=0))
        rows.append((s, G[:, -1].mean(), G[:, -1].min(), 100 * np.mean(np.isclose(G[:, -1], 0))))
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.2))
    a1.boxplot(finals); a1.set_xticks(range(1, len(STRATEGIES) + 1), [s.replace(" ", "\n", 1) for s in STRATEGIES])
    a1.set_ylabel("final gap to optimum (%)"); a1.set_title(f"{seeds} runs x {iters} iterations", loc="left")
    for s, c in zip(STRATEGIES, curves):
        a2.plot(c, label=s, lw=2)
    a2.set_xlabel("iteration"); a2.set_ylabel("mean gap of best found (%)"); a2.legend(fontsize=8)
    a2.set_title("average convergence", loc="left")
    plt.show()
    display(pd.DataFrame(rows, columns=["Strategy", "mean gap %", "best gap %", "% runs optimal"]).round(2))

## 4. Knapsack: escape strategies in action

### 4.1 Explorer
Each **iteration** (one point on the x-axis) is one cycle *escape move → local search → accept or not*.

**Parameters**

| Control | What it means | Used by |
|---|---|---|
| **instance** | The knapsack to solve. *Hiker (15 items)* is the slides example (greedy + LS = 67, optimum = 71). *Random (40 items)* is harder (greedy + LS = 567, optimum = 591). | all |
| **LS neighborhood** | The moves local search may use after each escape. *1-flip*: add or remove one item (only adding can improve). *1-flip + swap*: also exchange one packed item for an unpacked one. A larger neighborhood has fewer local optima, but each local-search step checks many more neighbors ($n$ vs. $n^2$). | all |
| **strategy** | How the search leaves the current local optimum (see the table in Section 3). | — |
| **k (strength)** | How much the escape move changes the solution. *Perturbation*: number of items flipped at random (the knapsack is then repaired if it is overweight). *Destroy & Repair*: number of packed items removed before the greedy refill. Small `k`: local search often falls back into the same local optimum. Large `k`: almost a random restart. | Perturbation, Destroy & Repair |
| **q (accept-worse prob.)** | After the worse move and the local search, a new local optimum that is **better or equal** is always accepted; a **worse** one is accepted with probability `q`. `q = 0`: never move to a worse local optimum. `q = 1`: always move (a random walk between local optima). | Accept worse |
| **iterations** | The computing budget: how many escape + local search cycles are run. | all |
| **random seed** | Fixes the random numbers. Same seed and same parameters give exactly the same run; change it to see another run of the same algorithm. | all except deterministic parts |

Multi-start ignores `k` and `q`; Perturbation and Destroy & Repair ignore `q`; Accept worse ignores `k` (it always makes **one** worse move).

**Reading the figure.** **Grey dots**: value right after the escape move, before local search (often much worse).
**Blue line**: the local optimum the search is currently standing on (it drops when a worse solution is accepted, and jumps around in Multi-start).
**Red line**: best solution found so far (it never gets worse). **Green dashed line**: the optimum.

**Reading the summary.** *gap to optimum* = $|\text{best} - \text{optimum}| / \text{optimum}$ in %; *found at iteration* = when the best value was first reached;
*distinct local optima visited* = how many different solutions the blue line stood on; *accepted moves* = how often the new local optimum replaced the current one.

In [ ]:
# @title Knapsack explorer { display-mode: "form" }
def kp_explorer(instance, neighborhood, strategy, k, q, iters, seed):
    T = cached_run(instance, neighborhood, strategy, iters, k, q, seed)
    fig, ax = plt.subplots(figsize=(11, 4.2))
    plot_trajectory(T, OPT[instance], ax, title=f"{strategy}: {STRATEGIES[strategy]}")
    ax.set_ylabel("total benefit")
    plt.show()
    summary(T, OPT[instance])

W.interact(kp_explorer,
           instance=dropdown(["Hiker (15 items)", "Random (40 items)"], "Random (40 items)", "instance"),
           neighborhood=dropdown(["1-flip", "1-flip + swap"], "1-flip", "LS neighborhood"),
           strategy=dropdown(list(STRATEGIES), "Perturbation (ILS)", "strategy"),
           k=slider(1, 10, 2, desc="k (strength)"),
           q=slider(0.0, 1.0, 0.3, 0.05, "q (accept-worse prob.)"),
           iters=slider(10, 200, 60, 10, "iterations"),
           seed=slider(0, 30, 0, desc="random seed"))

### 4.2 Fair comparison
One run proves nothing: randomness matters. Click **Run Interact** to run every strategy several times with the same budget.

* **runs (seeds)**: number of independent runs per strategy (seeds 0, 1, …, runs − 1). More runs give a more reliable comparison but take longer.
* The other controls have the same meaning as in 4.1 and apply to **all** strategies at once.
* **Left plot**: distribution of the final gap over the runs (box = middle 50% of the runs, line = median). **Right plot**: average gap of the best solution found, iteration by iteration. The faster and lower the curve, the better.
* **% runs optimal**: share of runs that found the optimum.

In [ ]:
# @title Knapsack: fair comparison { display-mode: "form" }
W.interact_manual(compare_strategies,
                  instance=dropdown(["Hiker (15 items)", "Random (40 items)"], "Random (40 items)", "instance"),
                  neighborhood=dropdown(["1-flip", "1-flip + swap"], "1-flip", "LS neighborhood"),
                  iters=slider(10, 200, 60, 10, "iterations"),
                  seeds=slider(5, 40, 20, desc="runs (seeds)"),
                  k=slider(1, 10, 2, desc="k (strength)"),
                  q=slider(0.0, 1.0, 0.3, 0.05, "q (accept-worse prob.)"))

## 5. p-Median: traps on a map

### 5.1 The p-median toolbox
Open exactly $p$ facilities; every customer is served by its **nearest** open facility; minimize the total cost.
A solution is the list of open facilities.

* **construction**: `greedy` (open, one by one, the facility that reduces the cost the most) and `random_start`
* **local search**: best-improvement **swap** (close one open facility, open a closed one), evaluated quickly with each customer's nearest and second-nearest open facility
* **escape moves**: `perturb` (*k* random swaps), `destroy_repair` (close *k* facilities, reopen greedily), `worse_neighbor` (one random worsening swap)
* `optimal`: brute force on small instances, an exact MILP (SciPy/HiGHS) on larger ones, **only** as a reference value

In [ ]:
# @title p-Median toolbox: greedy, swap local search and escape moves (worth reading) { display-mode: "form" }
class PMedianProblem:
    """p-median: open exactly p sites to minimize the total customer-to-nearest-open-site cost.
    A solution is a sorted numpy array with the p open sites."""
    sense = "min"

    def __init__(self, name, cost, p, customers_xy=None, sites_xy=None):
        self.name, self.D, self.p = name, np.asarray(cost, dtype=float), p
        self.n, self.m = self.D.shape
        self.cxy, self.sxy = customers_xy, sites_xy

    def value(self, S): return float(self.D[:, S].min(axis=1).sum())
    def describe(self, S): return "{" + ", ".join(f"F{j+1}" for j in S) + "}"

    def greedy(self, banned=(), S=None):
        """Open, one at a time, the site that reduces the total cost the most."""
        S = [] if S is None else list(S)
        while len(S) < self.p:
            cur = self.D[:, S].min(axis=1) if S else np.full(self.n, np.inf)
            cand = [j for j in range(self.m) if j not in S and j not in banned]
            costs = [np.minimum(cur, self.D[:, j]).sum() for j in cand]
            S.append(cand[int(np.argmin(costs))])
        return np.array(sorted(S))

    def random_start(self, rng):
        return np.sort(rng.choice(self.m, size=self.p, replace=False))

    def swap_costs(self, S):
        """Matrix [i, j] = total cost after closing S[i] and opening site outs[j]."""
        outs = np.setdiff1d(np.arange(self.m), S)
        DS = self.D[:, S]
        order = np.argsort(DS, axis=1)
        first = DS[np.arange(self.n), order[:, 0]]
        second = DS[np.arange(self.n), order[:, 1]] if len(S) > 1 else np.full(self.n, np.inf)
        res = np.empty((len(S), outs.size))
        for i in range(len(S)):
            base = np.where(order[:, 0] == i, second, first)   # cost if S[i] is closed
            res[i] = np.minimum(base[:, None], self.D[:, outs]).sum(axis=0)
        return res, outs

    def best_move(self, S):
        res, outs = self.swap_costs(S)
        i, j = np.unravel_index(np.argmin(res), res.shape)
        if res[i, j] < self.value(S) - 1e-9:
            return ("swap", S[i], outs[j])
        return None

    def apply(self, S, move):
        _, a, b = move
        return np.array(sorted([s for s in S if s != a] + [b]))

    def local_search(self, S, path=None):
        S = np.array(S)
        while (move := self.best_move(S)) is not None:
            S = self.apply(S, move)
            if path is not None: path.append(S.copy())
        return S

    def perturb(self, S, k, rng):
        """k random swaps."""
        S = list(S)
        for _ in range(k):
            out = rng.choice(S)
            new = rng.choice(np.setdiff1d(np.arange(self.m), S))
            S = [s for s in S if s != out] + [int(new)]
        return np.array(sorted(S))

    def destroy_repair(self, S, k, rng):
        """Close k random facilities, reopen greedily (closed ones are banned during the repair)."""
        out = rng.choice(S, size=min(k, len(S)), replace=False)
        keep = [s for s in S if s not in out]
        return self.greedy(banned=set(out.tolist()), S=keep)

    def worse_neighbor(self, S, rng):
        """A deliberately worse swap: pick a random swap that increases the cost."""
        res, outs = self.swap_costs(S)
        worse = np.argwhere(res > self.value(S) + 1e-9)
        if len(worse) == 0: return self.perturb(S, 1, rng)
        i, j = worse[rng.integers(len(worse))]
        return self.apply(S, ("swap", S[i], outs[j]))

    def optimal(self):
        if math.comb(self.m, self.p) <= 200_000:
            return np.array(min(itertools.combinations(range(self.m), self.p), key=lambda S: self.value(list(S))))
        # Larger instances: exact MILP with SciPy/HiGHS (reference value only, not part of the heuristics)
        from scipy.optimize import milp, LinearConstraint, Bounds
        from scipy.sparse import lil_matrix
        n, m = self.n, self.m
        nv = m + n * m                     # y_j (open site j), then x_ij (customer i served by j)
        c = np.concatenate([np.zeros(m), self.D.ravel()])
        A = lil_matrix((n + n * m + 1, nv)); lo, hi = [], []
        for i in range(n):                 # every customer served exactly once
            A[i, m + i * m: m + (i + 1) * m] = 1; lo.append(1); hi.append(1)
        r = n
        for i in range(n):                 # x_ij <= y_j
            for j in range(m):
                A[r, m + i * m + j] = 1; A[r, j] = -1; lo.append(-np.inf); hi.append(0); r += 1
        A[r, :m] = 1; lo.append(self.p); hi.append(self.p)   # exactly p open sites
        integrality = np.concatenate([np.ones(m), np.zeros(n * m)])
        res = milp(c, constraints=LinearConstraint(A.tocsr(), lo, hi), integrality=integrality, bounds=Bounds(0, 1))
        return np.flatnonzero(res.x[:m] > 0.5)


In [ ]:
# @title Map instances and drawing helpers { display-mode: "form" }
def make_map_instance(name, seed, n, p, clusters):
    """n towns in clusters on a 100x100 map; every town is a customer and a candidate facility site."""
    rng = np.random.default_rng(seed)
    centers = rng.uniform(10, 90, (clusters, 2))
    xy = np.vstack([rng.normal(centers[rng.integers(clusters)], 8) for _ in range(n)]).clip(0, 100)
    D = np.linalg.norm(xy[:, None] - xy[None], axis=2).round(1)
    return PMedianProblem(name, D, p, xy, xy)

def plot_map(prob, S, ax, title="", ref=None):
    S = np.asarray(S)
    a = np.argmin(prob.D[:, S], axis=1)
    cmap = plt.get_cmap("tab20")
    for i in range(prob.n):
        f = S[a[i]]
        ax.plot([prob.cxy[i, 0], prob.sxy[f, 0]], [prob.cxy[i, 1], prob.sxy[f, 1]], color=cmap(a[i] % 20), lw=0.7, alpha=0.6)
    ax.scatter(prob.cxy[:, 0], prob.cxy[:, 1], s=14, color=[cmap(v % 20) for v in a], zorder=2)
    ax.scatter(prob.sxy[S, 0], prob.sxy[S, 1], marker="s", s=80, color="k", zorder=3, label="open facility")
    if ref is not None:
        ax.scatter(prob.sxy[ref, 0], prob.sxy[ref, 1], s=300, facecolor="none", edgecolor=COLORS["opt"], lw=2.2,
                   zorder=4, label="optimal facility")
    ax.set_title(f"{title}cost = {prob.value(S):.1f}", fontsize=10)
    ax.set_xlim(-3, 103); ax.set_ylim(-3, 103)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)

### 5.2 The slides instance: 8 sites, 10 customers, $p = 3$

In [ ]:
# @title Slides instance: greedy, swap local search and basins { display-mode: "form" }
SLIDES_D = np.array([[23,  8, 15, 21, 23, 20, 18, 21], [21, 15, 18, 25,  5,  9, 22, 21],
                     [ 7,  9, 14, 23,  7, 15, 14, 22], [ 8,  7,  7,  7, 18,  5,  6, 24],
                     [ 6, 18, 12, 22,  6, 10, 20, 14], [22, 23, 19,  7, 19, 21, 17, 17],
                     [17, 10, 11, 15, 20,  6, 10,  8], [ 5,  8, 10, 22, 13, 12, 23, 18],
                     [20, 20, 15, 22,  7, 18, 24, 11], [ 8,  8, 13, 17, 21,  8, 15,  9]])
PM_SLIDES = PMedianProblem("Warehouses (slides)", SLIDES_D, p=3)

g = PM_SLIDES.greedy()
path = [g]; ls = PM_SLIDES.local_search(g, path)
print("Greedy        :", PM_SLIDES.describe(g), "cost", PM_SLIDES.value(g))
for a, b in zip(path, path[1:]):
    print("  swap ->     :", PM_SLIDES.describe(b), "cost", PM_SLIDES.value(b))
print("Optimum       :", PM_SLIDES.describe(PM_SLIDES.optimal()), "cost", PM_SLIDES.value(PM_SLIDES.optimal()))

ends = Counter(tuple(PM_SLIDES.local_search(np.array(S))) for S in itertools.combinations(range(PM_SLIDES.m), PM_SLIDES.p))
print(f"\nSwap local search started from each of the {sum(ends.values())} possible solutions ends in:")
for S, c in ends.items():
    print(f"  {PM_SLIDES.describe(np.array(S))} (cost {PM_SLIDES.value(list(S)):g}) from {c} starts")

> On this small instance there is **a single basin of attraction**: wherever you start, swap local search reaches the
> optimum (73). No escape is needed. Real instances are rarely this kind, as the next map shows.

### 5.3 Twelve towns, three depots: a landscape with a trap
All $\binom{12}{3} = 220$ solutions can be enumerated. For each one, we run swap local search and record where it ends.

In [ ]:
# @title Enumerate the 220 solutions and their basins { display-mode: "form" }
PM_SMALL = make_map_instance("12 towns, 3 depots", seed=9, n=12, p=3, clusters=4)
SMALL_COMBOS = list(itertools.combinations(range(PM_SMALL.m), PM_SMALL.p))
SMALL_END = {S: tuple(PM_SMALL.local_search(np.array(S))) for S in SMALL_COMBOS}
BASINS = Counter(SMALL_END.values())
SMALL_LO = sorted(BASINS, key=lambda S: PM_SMALL.value(list(S)))
SMALL_OPT = PM_SMALL.value(list(SMALL_LO[0]))
greedy_end = tuple(PM_SMALL.local_search(PM_SMALL.greedy()))

fig, axes = plt.subplots(1, len(SMALL_LO) + 1, figsize=(4.2 * (len(SMALL_LO) + 1), 4))
labels = [f"LO{i + 1}\n{PM_SMALL.value(list(S)):.1f}" for i, S in enumerate(SMALL_LO)]
cols = [COLORS["opt"] if np.isclose(PM_SMALL.value(list(S)), SMALL_OPT) else COLORS["trap"] for S in SMALL_LO]
axes[0].bar(labels, [BASINS[S] for S in SMALL_LO], color=cols)
axes[0].set_ylabel("number of starting solutions"); axes[0].set_title("Basin of attraction of each local optimum", fontsize=10)
for i, S in enumerate(SMALL_LO):
    tag = "  <- greedy + LS ends here" if S == greedy_end else ""
    plot_map(PM_SMALL, list(S), axes[i + 1], title=f"LO{i + 1} {PM_SMALL.describe(np.array(S))}{tag}\n")
plt.tight_layout(); plt.show()
print(f"{len(SMALL_LO)} local optima. Greedy + swap LS ends at cost {PM_SMALL.value(list(greedy_end)):.1f}; "
      f"the optimum is {SMALL_OPT:.1f}.")

**Try it.** Pick a starting solution (a slider over the 220 possibilities) and watch where swap local search takes it.
How often do you fall in the trap?

In [ ]:
# @title Interactive basin explorer { display-mode: "form" }
def small_basin(start):
    S0 = np.array(SMALL_COMBOS[start])
    path = [S0]; S1 = PM_SMALL.local_search(S0, path)
    trapped = not np.isclose(PM_SMALL.value(S1), SMALL_OPT)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4.3))
    plot_map(PM_SMALL, S0, a1, title=f"start {PM_SMALL.describe(S0)}\n")
    plot_map(PM_SMALL, S1, a2, title=f"after {len(path) - 1} swap(s): {PM_SMALL.describe(S1)}"
                                     f"{'  TRAPPED' if trapped else '  optimal'}\n", ref=list(SMALL_LO[0]))
    plt.tight_layout(); plt.show()
    for a, b in zip(path, path[1:]):
        print(f"close F{list(set(a) - set(b))[0] + 1}, open F{list(set(b) - set(a))[0] + 1} -> cost {PM_SMALL.value(b):.1f}")

W.interact(small_basin, start=slider(0, len(SMALL_COMBOS) - 1, SMALL_COMBOS.index(tuple(PM_SMALL.greedy())), desc="start #"))

## 6. p-Median: escape strategies in action

### 6.1 A bigger map: 150 customers, 12 facilities

In [ ]:
# @title Build the 150-customer map and its optimum { display-mode: "form" }
PM_MAP = make_map_instance("Map (150 customers, p=12)", seed=3, n=150, p=12, clusters=10)
try:
    PM_MAP_OPT = PM_MAP.optimal()                # exact MILP with SciPy (a few seconds at most)
    opt_label = "optimum"
except Exception as err:                         # no SciPy: use the best of a long ILS run as a reference
    print("Exact solver not available:", err)
    PM_MAP_OPT = run_escape(PM_MAP, "Perturbation (ILS)", 400, k=2, seed=0)["best_sol"]
    opt_label = "best known"
INSTANCES[PM_MAP.name], OPT[PM_MAP.name] = PM_MAP, PM_MAP.value(PM_MAP_OPT)

g = PM_MAP.greedy(); ls = PM_MAP.local_search(g)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, S, name in zip(axes, [g, ls, PM_MAP_OPT], ["Greedy", "Greedy + swap LS", opt_label.capitalize()]):
    plot_map(PM_MAP, S, ax, title=f"{name}: gap {gap(PM_MAP.value(S), OPT[PM_MAP.name]):.2f}%\n", ref=PM_MAP_OPT)
axes[0].legend(loc="lower left", fontsize=8)
plt.tight_layout(); plt.show()

### 6.2 Explorer: replay the search on the map
Same idea as the knapsack explorer, with p-median moves.

| Control | What it means | Used by |
|---|---|---|
| **strategy** | How the search leaves the current local optimum (Section 3). | — |
| **k (strength)** | *Perturbation*: number of random swaps (close a random open facility, open a random closed site). *Destroy & Repair*: number of open facilities closed and then reopened greedily (the closed ones cannot be reopened during that repair). | Perturbation, Destroy & Repair |
| **q (accept-worse prob.)** | Probability of accepting a **worse** local optimum. The worse move is one random swap that increases the cost. | Accept worse |
| **iterations** | Number of escape + local search cycles. | all |
| **random seed** | Fixes the random numbers (same seed = same run). | all |
| **replay step** | Which iteration is displayed: the **map** shows the *current* solution at that iteration and the vertical line marks it on the chart. Values above `iterations` show the last iteration. | — |

On the map, black squares are open facilities, each customer is linked to its nearest open facility, and **green circles** mark the optimal facilities: when every square sits in a circle, the search has found the optimum.
Here costs are **minimized**, so improvements make the curves go **down**.

In [ ]:
# @title p-median explorer { display-mode: "form" }
def pm_explorer(strategy, k, q, iters, seed, step):
    T = cached_run(PM_MAP.name, "-", strategy, iters, k, q, seed)
    step = min(step, iters)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5.2), gridspec_kw={"width_ratios": [1, 1.4]})
    plot_map(PM_MAP, T["sols"][step], a1, title=f"iteration {step}: current solution, ", ref=PM_MAP_OPT)
    plot_trajectory(T, OPT[PM_MAP.name], a2, upto=step, title=f"{strategy}: {STRATEGIES[strategy]}")
    a2.set_ylabel("total cost (lower is better)")
    plt.tight_layout(); plt.show()
    summary(T, OPT[PM_MAP.name])

W.interact(pm_explorer,
           strategy=dropdown(list(STRATEGIES), "Destroy & Repair", "strategy"),
           k=slider(1, 8, 2, desc="k (strength)"),
           q=slider(0.0, 1.0, 0.3, 0.05, "q (accept-worse prob.)"),
           iters=slider(10, 100, 40, 10, "iterations"),
           seed=slider(0, 30, 0, desc="random seed"),
           step=slider(0, 100, 40, desc="replay step"))

### 6.3 Fair comparison
Same as 4.2 for the p-median map: every strategy is run `runs` times with the same parameters, and the gap is measured against the optimal cost.

In [ ]:
# @title p-median: fair comparison { display-mode: "form" }
def pm_compare(iters, seeds, k, q):
    compare_strategies(PM_MAP.name, "-", iters, seeds, k, q)

W.interact_manual(pm_compare,
                  iters=slider(10, 100, 40, 10, "iterations"),
                  seeds=slider(5, 30, 10, desc="runs (seeds)"),
                  k=slider(1, 8, 2, desc="k (strength)"),
                  q=slider(0.0, 1.0, 0.3, 0.05, "q (accept-worse prob.)"))

## 7. Takeaways and exercises

**Takeaways**
1. A local optimum is defined **by a neighborhood**: what is a trap for 1-swap moves may not be one for 2-swap moves (startups, hiker + swap).
2. Every escape strategy has the same shape: **escape → local search → accept?**, and we always keep the best solution found.
3. Escapes that **keep part of the current solution** (perturbation, destroy & repair) usually beat restarting from scratch (multi-start).
4. Strength matters: too small and local search falls back into the same local optimum; too large and the search becomes a random restart.
5. Accepting worse moves **blindly** makes the search wander (and drift downhill). It needs control: a schedule (Simulated Annealing) or a memory (**Tabu Search**, next session).

**Exercises** (answer in a markdown cell below each one)
1. *Landscape (2.2).* From {A, B}, which removals lead to the global optimum, and which lead to another trap? Explain with the figure.
2. *Neighborhoods (2.3).* With the `1-flip + swap` neighborhood, does greedy + LS reach the optimum on the hiker instance? What do you pay for the bigger neighborhood?
3. *Strength (4.1).* On *Random (40 items)* with *Perturbation*, try `k = 1, 3, 6, 10`. Which `k` works best? Relate your answer to the "Key trade-off" slide.
4. *Accept worse (4.1).* Set `q = 0` and then `q = 1`. How many distinct local optima are visited? Does visiting more of them help? Why is uncontrolled acceptance not enough? (This is the motivation for the next session: **Tabu Search**.)
5. *p-median (6.2).* Compare *Destroy & Repair* with `k = 1` and `k = 6`. Replay the run on the map: which facilities move?
6. *Coding challenge.* Write a "worst destroy" for p-median: close the facilities whose removal increases the cost the least, and compare it with the random destroy.